# Tutorial 10-2: Hard vs. Soft Clustering (k-means vs. GMM)
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

---
## 1. Introduction: The Concept of 'Soft' Assignment
In the first part of this lecture, we looked at **k-means**, which performs **Hard Clustering**. Every data point is assigned to exactly one cluster centroid. While efficient, this doesn't account for uncertainty—if a point sits exactly between two centroids, k-means still forces it into one group.

**Soft Clustering**, specifically using **Gaussian Mixture Models (GMM)** and the **Expectation-Maximization (EM)** algorithm, treats each cluster as a probability distribution (a Gaussian). Instead of a label, every point receives a 'responsibility' score ($w_j^{(i)}$) representing the probability that it belongs to each cluster.

### The Objective
In this tutorial, we will:
1. Generate a dataset with overlapping clusters.
2. Visualize the rigid 'Voronoi' boundaries created by k-means.
3. Compare this to the probabilistic density regions and soft assignments of a GMM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs
from matplotlib.patches import Ellipse

# Step 1: Generate synthetic data that intentionally overlaps
X, y_true = make_blobs(n_samples=400, centers=3, cluster_std=0.80, random_state=0)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=30, color='gray', alpha=0.5)
plt.title("Overlapping Synthetic Clusters")
plt.show()

## 2. Hard Clustering with k-means
As we saw in Tutorial 10-1, k-means partitions the space into regions. The boundary between any two clusters is always a straight line (in 2D) or a hyperplane (in higher dimensions). These are known as **Voronoi Cells**.

Notice in the plot below how points on the very edge of a cluster are treated with the same 'certainty' as points at the center.

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = kmeans.fit_predict(X)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=labels, s=30, cmap='viridis', zorder=2)

# Plot the decision boundaries (Voronoi regions)
h = .02
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = kmeans.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.2, cmap='viridis')
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            s=200, marker='X', c='red', label='Centroids')
plt.title("k-means: Hard Assignments and Voronoi Boundaries")
plt.legend()
plt.show()

## 3. Soft Clustering with Gaussian Mixture Models (GMM)
The GMM assumes that data is generated from a mixture of $k$ Gaussian distributions. It uses the **EM Algorithm** to estimate the parameters:
* **E-step:** Calculate the 'soft' probability ($w_j^{(i)}$) that point $i$ belongs to cluster $j$.
* **M-step:** Update the means ($\mu$), covariances ($\Sigma$), and mixture weights ($\phi$) based on these probabilities.

In the plot below, the points are colored by their most likely cluster, but notice how the 'faded' points on the boundaries indicate lower probability (uncertainty).

In [ ]:
gmm = GaussianMixture(n_components=3, random_state=0)
gmm.fit(X)
probs = gmm.predict_proba(X)

# Determine the color based on the probabilities
# This creates a visual 'blend' where points with high uncertainty appear greyish/mixed
colors = probs

plt.figure(figsize=(10, 8))
plt.scatter(X[:, 0], X[:, 1], c=colors, s=30, edgecolors='black', linewidth=0.5)

# Function to draw density ellipses
def draw_ellipse(position, covariance, ax=None, **kwargs):
    ax = ax or plt.gca()
    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width, height = 2 * np.sqrt(covariance)
    for nsig in range(1, 4):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height,
                             angle, **kwargs))

for i in range(3):
    draw_ellipse(gmm.means_[i], gmm.covariances_[i], alpha=0.1, color='blue')

plt.title("GMM: Soft Assignments and Density Ellipses\n(Colors represent the probability distribution across clusters)")
plt.show()

## 4. Comparing the Membership 'Confidence'
Let's pick a specific point that sits between two clusters. In k-means, it would have one label. In GMM, we can see exactly how much 'responsibility' each cluster takes for that point.

In [ ]:
# Find a point with high uncertainty (probabilities are somewhat balanced)
uncertain_idx = np.where((probs[:, 0] > 0.3) & (probs[:, 1] > 0.3))[0][0]
point = X[uncertain_idx]
point_probs = probs[uncertain_idx]

print(f"For point at coordinates {point}:")
print(f"k-means Label: {labels[uncertain_idx]}")
print(f"GMM Probabilities: Cluster 0: {point_probs[0]:.2f}, Cluster 1: {point_probs[1]:.2f}, Cluster 2: {point_probs[2]:.2f}")

## Summary

### 1. Hard vs. Soft
* **k-means (Hard):** Every point is an 'either-or'. It minimizes the squared distance to centroids. It works best for clearly separated, spherical clusters of similar size.
* **GMM (Soft):** Every point has a degree of membership. It models the data as a mixture of densities. It is much more flexible and can handle overlapping data or clusters with different shapes/orientations.

### 2. The Power of EM
The EM algorithm is a powerful general-purpose tool for density estimation. By introducing 'Latent Variables' (the hidden cluster memberships), it allows us to optimize complex likelihood functions that would otherwise be impossible to solve directly.

### 3. When to use which?
If you need a fast, simple grouping and your data is well-behaved, k-means is the standard choice. If you need to understand the underlying probability distribution or your clusters are significantly overlapping, GMM provides a more natural and insightful solution.